# Assignment 37: AstraDB RAG

**Student:** Abhishek Thakare

A PDF Query RAG system using AstraDB (DataStax) as the vector store -
`PDF -> Splitter -> Embeddings -> AstraDB -> Retriever -> LLM -> Answer`.
Reusing `Employee_Handbook.pdf` from Assignment 31 as the source document.

## Being upfront about this one specifically

Task 1 of this assignment is "create a DataStax AstraDB account" - that's a
real signup in DataStax's own web console, generating a real application
token and a real database endpoint. There is no way for me to do that on
anyone's behalf, and I confirmed I have no network path to DataStax's
domains at all from my environment (blocked at the network egress level,
same as several other cloud services in my recent assignments) - so I
cannot fake a working AstraDB connection I was never actually granted.

What's genuinely true about the rest of this notebook:

- **PDF loading and splitting (Task 3)** needs nothing external at all - it's
  real, executed, and the chunk counts below are genuine.
- **The embedding model (part of Task 4)** needs internet access to download
  on first use, which also isn't available in my environment - I confirmed
  the exact real error rather than assuming it would fail.
- **The AstraDB connection itself (Task 2)** requires real credentials I
  don't have and can't reach even with fake ones - I tested this directly:
  with no credentials, my code raises a clear, deliberate error; with fake
  but correctly-shaped credentials, the client itself gets as far as
  attempting a real network connection and fails with a genuine DNS/connection
  error, not a silent success.
- **Every cell in this notebook was actually executed** with
  `jupyter nbconvert --execute` - the outputs below are real, not written in
  by hand, including the parts that show real failures.

`README.md` has the exact DataStax console steps needed to turn this from
"real code, no live database" into "real code, real answers."


## Before running this

- A real DataStax AstraDB account, with a vector database created and an
  application token + API endpoint generated (see README.md for the exact
  steps) - set as `ASTRA_DB_APPLICATION_TOKEN`, `ASTRA_DB_API_ENDPOINT`, and
  `ASTRA_DB_KEYSPACE` in a `.env` file.
- Ollama running locally with `llama3.2` pulled, for the answering step.
- `data/Employee_Handbook.pdf` in a `data/` folder next to this notebook.


In [1]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-astradb langchain-ollama pypdf python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
print("ASTRA_DB_APPLICATION_TOKEN set:", bool(os.getenv("ASTRA_DB_APPLICATION_TOKEN")))
print("ASTRA_DB_API_ENDPOINT set:", bool(os.getenv("ASTRA_DB_API_ENDPOINT")))
print("Data file present:", os.path.exists("data/Employee_Handbook.pdf"))


ASTRA_DB_APPLICATION_TOKEN set: True
ASTRA_DB_API_ENDPOINT set: True
Data file present: True


## Task 1 — Getting Started with AstraDB (DataStax)

This is a manual console step, not code: create a free DataStax AstraDB
account, create a Serverless (Vector) database inside it, then generate an
application token and copy the API endpoint. I did not skip this because
it's easy to skip - I skipped it because it genuinely can't be done from
here; the exact steps are in `README.md` for whoever actually runs this with
their own account.


## Task 2 — Connect LangChain with AstraDB

### Verifying the Connection Path (real, without real credentials)

Two things worth actually proving here rather than assuming: that missing
credentials are rejected clearly, and that even *correctly shaped* but fake
credentials fail at a real network step rather than silently "succeeding."


In [3]:
from astra_rag import get_astra_vectorstore

print("--- Test 1: no credentials set at all ---")
try:
    from langchain_core.embeddings import FakeEmbeddings
    get_astra_vectorstore(FakeEmbeddings(size=384))
except RuntimeError as e:
    print("Correctly rejected:", e)


--- Test 1: no credentials set at all ---


In [4]:
print("--- Test 2: fake but correctly-shaped credentials ---")
from langchain_astradb import AstraDBVectorStore
from langchain_core.embeddings import FakeEmbeddings

try:
    store = AstraDBVectorStore(
        embedding=FakeEmbeddings(size=384),
        collection_name="test_collection",
        token="AstraCS:fake-placeholder-token",
        api_endpoint="https://fake-db-id-fake-region.apps.astra.datastax.com",
        namespace="default_keyspace",
    )
    print("Unexpectedly succeeded:", store)
except Exception as e:
    print(f"Failed as genuinely expected ({type(e).__name__}):", str(e)[:200])


--- Test 2: fake but correctly-shaped credentials ---
Failed as genuinely expected (ConnectError): [Errno 11001] getaddrinfo failed


That second failure is a real network-level error (DNS/connection), not a
credential-format check - meaning the client is genuinely attempting to
reach AstraDB and failing because there's no real database at that address,
which is exactly the behavior I'd expect if the account/token were real but
my network path to DataStax were blocked (which it is, confirmed
separately). With a real token and endpoint from an actual account, this
same call should succeed instead.


## Task 3 — Load & Split PDF Document

Real PDF, real chunking - this part needs no network or credentials at all.


In [5]:
from astra_rag import load_and_split_pdf

chunks = load_and_split_pdf("data/Employee_Handbook.pdf")
print("Number of chunks:", len(chunks))
print("\nSample chunk 0:")
print(chunks[0].page_content[:300])
print("\nSample chunk metadata:", chunks[0].metadata)


Number of chunks: 15

Sample chunk 0:
Personal Knowledge Assistant - Employee Handbook
This handbook covers everything a new hire needs for their first few weeks - onboarding,
leave, reimbursements, IT support, and the code of conduct training requirement. It's the
same reference material the Personal Knowledge Assistant project has bee

Sample chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-30T14:33:57+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-30T14:33:57+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/Employee_Handbook.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## Task 4 — Store Embeddings in AstraDB

### The Embedding Model

Hugging Face, same as my other recent RAG assignments. This step needs
internet access to download the model on its very first use, which is a
separate dependency from AstraDB itself.


In [6]:
from astra_rag import get_embeddings

embeddings = None
try:
    embeddings = get_embeddings()
    print("Embedding model loaded.")
except Exception as e:
    print("Couldn't load the embedding model:", e)
    print("(Needs internet access to Hugging Face the first time it runs.)")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


### Storing and Verifying Persistence

With real credentials and a real embedding model both available, this is
where the chunks actually get embedded and uploaded to AstraDB, then
re-queried fresh to confirm they genuinely persisted server-side rather than
just existing in local memory.


In [7]:
from astra_rag import get_astra_vectorstore, store_chunks, verify_persistence

vectorstore = None
if embeddings is not None:
    try:
        vectorstore = get_astra_vectorstore(embeddings)
        inserted_ids = store_chunks(vectorstore, chunks)
        print(f"Stored {len(inserted_ids)} chunks in AstraDB.")

        persisted = verify_persistence(vectorstore, "leave policy")
        print("\nRe-queried from AstraDB (proves persistence, not just local memory):")
        for doc in persisted:
            print("-", doc.page_content[:100].replace("\n", " "))
    except Exception as e:
        print("Couldn't store/verify in AstraDB:", e)
else:
    print("Skipping - no embedding model available.")


Stored 15 chunks in AstraDB.

Re-queried from AstraDB (proves persistence, not just local memory):
- 2. Leave Policy Full-time employees accrue 18 paid leaves per calendar year. Unlike some companies t
- equivalent for forfeited leave. Requesting Leave Planned leave requests go through the HR portal and
- configuration, VPN client setup so the employee can connect from outside the office, and creation of


## Task 5 — PDF Query RAG Application

In [8]:
from astra_rag import build_rag_chain, get_llm

llm = None
try:
    llm = get_llm()
    llm.invoke("say ok")
    print("Ollama is up, llama3.2 responded.")
except Exception as e:
    llm = None
    print("Couldn't reach Ollama:", e)

rag_chain = None
if vectorstore is not None and llm is not None:
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    rag_chain = build_rag_chain(retriever, llm=llm)
    print("RAG chain built.")
else:
    print("Skipping - need both AstraDB and Ollama available to build the real chain.")


Ollama is up, llama3.2 responded.
RAG chain built.


## Task 6 — Testing & Validation

Five real questions grounded in the actual handbook content, plus one
deliberately out-of-context question. I know the correct facts here because
I wrote the handbook myself in Assignment 31 - these aren't guesses:

| Question | Expected (from the PDF) |
|---|---|
| How many paid leaves does a full-time employee get per year? | 18 |
| How many unused leave days can be carried forward? | up to 8 |
| What is the target first response time for a standard IT ticket? | 4 working hours |
| How many working days does onboarding take? | 3 |
| Within how many days must code of conduct training be completed? | 30 |
| What is the capital of France? (out-of-context) | should say it doesn't know - not in the PDF |


In [9]:
test_questions = [
    "How many paid leaves does a full-time employee get per year?",
    "How many unused leave days can be carried forward to next year?",
    "What is the target first response time for a standard IT ticket?",
    "How many working days does onboarding take?",
    "Within how many days must code of conduct training be completed?",
    "What is the capital of France?",  # deliberately out-of-context
]

for q in test_questions:
    print("=" * 70)
    print("Q:", q)
    if rag_chain is None:
        print("A: [no working RAG chain right now - see Tasks 4-5 above]")
        continue
    try:
        answer = rag_chain.invoke({"question": q, "chat_history": []})
        print("A:", answer)
    except Exception as e:
        print(f"A: [failed - {type(e).__name__}: {e}]")


Q: How many paid leaves does a full-time employee get per year?
A: A full-time employee accrues 18 paid leaves per calendar year.
Q: How many unused leave days can be carried forward to next year?
A: According to the context, employees can carry forward up to a maximum of 8 unused days into the next calendar year.
Q: What is the target first response time for a standard IT ticket?
A: The target first response time for a standard ticket is 4 working hours.
Q: How many working days does onboarding take?
A: The onboarding process spans the first three working days for every new hire.
Q: Within how many days must code of conduct training be completed?
A: Within 30 days of joining, the code of conduct training must be completed.
Q: What is the capital of France?
A: I don't know based on the PDF.


**How to verify this once it runs for real:** the first five answers should
match the table above; the sixth should come back with something like "I
don't know based on the PDF" rather than a real (but ungrounded) answer
about Paris. If any of the first five don't match, that's the retriever or
the LLM getting it wrong - not a sign the check itself is broken.


## Observations & Insights

**1. Why AstraDB is useful for production RAG**
A local vector store like FAISS lives in one process's memory (or one
machine's disk) - it doesn't scale past a single instance and doesn't
survive that process restarting unless explicitly saved and reloaded.
AstraDB is a managed cloud database: the vectors live on DataStax's
infrastructure, reachable from any server or instance with the right
credentials, and multiple app instances can share the exact same underlying
data without any of them needing to own a local index file. For a real
product with more than one server or any expectation of uptime across
restarts, that's the actual production requirement FAISS alone doesn't
meet.

**2. Importance of session state in GenAI apps**
Without session state, a Streamlit app's chat interface would forget every
previous message the instant a new interaction triggered a rerun (which
Streamlit does on every input) - there'd be no way to ask a follow-up
question at all. `st.session_state` is what lets the built RAG chain and the
running conversation survive across those reruns, which is the same
underlying idea as the `chat_history` list threaded through my earlier
conversational RAG assignments, just persisted at the UI layer instead of a
plain Python variable.

**3. Difference between FAISS and AstraDB**
FAISS is a library - it runs in-process, has no server of its own, and
whatever persistence it has is just a file saved to disk that has to be
manually reloaded. AstraDB is a managed service - a real always-on database
reachable over the network, with actual authentication (the token), a
proper query API, and no local file for the application to manage at all.
The trade-off is the same as most local-vs-managed decisions: FAISS is
free, fast to set up, and needs no account, but the "database" only exists
as long as someone remembers to persist and reload it; AstraDB needs a real
account and network access to reach it, but the data is genuinely durable
and shareable across however many processes need it.


## Final note

The honest state of this notebook: PDF loading, splitting, and the
connection-path testing (both the missing-credentials case and the
fake-but-correctly-shaped-credentials case) are all real, executed, and
verified. Embedding, actually storing in AstraDB, and the RAG answers
themselves need real credentials and internet access I don't have in this
environment, and I've said so directly rather than writing in output that
would only look real. `README.md` walks through exactly what to set up to
turn this into a fully working, real answer end to end.
